# NB8 — XLM-RoBERTa as the 4th frozen probe (action #5)

**Why.** We picked CAMeLBERT-MSA as the neural core from NB5c (AraBERTv2 / AraELECTRA / CAMeLBERT-MSA).
But the literature flags a direct threat: the multilingual **XLM-R** often beats dedicated Arabic
encoders at AI-text detection (BUSTED@AraGenEval: XLM-R 0.7701 > AraELECTRA & CAMeLBERT; AbjadGenEval
dominated by XLM-R + DeBERTa-v3). An examiner will ask whether we tried it. This notebook adds XLM-R as
a 4th frozen probe under the NB5c protocol (std / LOGO-mean / LOGO-worst).

**We do NOT re-run AraBERTv2 or AraELECTRA** — their NB5c numbers stand and are shown for reference.
But to compare XLM-R against CAMeLBERT-MSA *fairly*, CAMeLBERT is **re-anchored on the SAME harness
here** (loaded from the cached embeddings — no rebuild), because a probe harness can differ by ~0.5pp
from NB5c. The decision that matters — *does XLM-R beat our chosen core?* — is then apples-to-apples.

Frozen probe = encoder frozen, [CLS]/<s> extracted, LogReg on top (raw embeddings, NB5c convention).
Single top-to-bottom pass. Only XLM-R embeddings are heavy (first run ~hrs on GPU); they cache to
/kaggle/working — promote to a dataset for minute-long re-runs.

## 1 · Config

In [1]:
import os, numpy as np, pandas as pd
P_DATASET   = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"                 # EDIT
# CAMeLBERT-MSA embeddings you already cached from NB6g (768-d, article_id + n0..n767):
P_CAMEL_EMB = "/kaggle/input/notebooks/bahaaqassem/nb6g-passive-ratio-decision/embeddings.parquet"  # EDIT to the exact Input-pane path
# XLM-R embeddings: point at a dataset if you've cached them; else built inline this run:
P_XLMR_EMB  = "/kaggle/input/aigt-xlmr-emb/xlmr_embeddings.parquet"
W_XLMR_EMB  = "/kaggle/working/xlmr_embeddings.parquet"

XLMR_MODEL = "xlm-roberta-base"                 # 768-d, matches CAMeLBERT dim
GENERATORS = ["deepseek", "sonnet", "qwen", "gemini", "gpt", "opus"]
EXPECT_CAMEL_CACHE = True                        # you have it -> fail fast if the path is wrong

# NB5c reference (frozen probe: std / LOGO-mean / LOGO-worst) — shown for context, NOT recomputed
NB5C = {"AraBERTv2":(99.5,97.6,92.8), "AraELECTRA":(99.5,96.2,83.5), "CAMeLBERT-MSA":(99.7,98.1,92.7)}
print("config loaded")

config loaded


## 2 · Load + align dataset

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
y   = df["label"].to_numpy()
gen = df["generator"].fillna("__human__").to_numpy()
is_train = (df["split"]=="train").to_numpy()
is_tv    = df["split"].isin(["train","val"]).to_numpy()
is_test  = (df["split"]=="test").to_numpy()
print(f"{len(df)} articles | label {dict(zip(*np.unique(y, return_counts=True)))} | "
      f"train {is_train.sum()} val {(df['split']=='val').sum()} test {is_test.sum()}")

7101 articles | label {np.int64(0): np.int64(3500), np.int64(1): np.int64(3601)} | train 5363 val 645 test 1093


## 3 · Load CAMeLBERT-MSA embeddings (re-anchor, from cache — no rebuild)

In [3]:
def load_emb(*paths):
    for p in paths:
        if p and os.path.exists(p):
            e = pd.read_parquet(p)
            if "article_id" in e.columns: e = e.set_index("article_id")
            return e.reindex(df.index)
    return None

camel = load_emb(P_CAMEL_EMB)
if camel is None and EXPECT_CAMEL_CACHE:
    raise FileNotFoundError(f"CAMeLBERT cache not found at P_CAMEL_EMB={P_CAMEL_EMB}\n"
                            "Copy the exact path from the Input pane, or set EXPECT_CAMEL_CACHE=False to skip re-anchor.")
CAMEL_COLS = [c for c in camel.columns if c != "article_id"]
assert len(CAMEL_COLS)==768, f"expected 768, got {len(CAMEL_COLS)}"
assert (camel.index==df.index).all(), "camel<->df alignment failed"
X_camel = camel[CAMEL_COLS].to_numpy(np.float32)
print("CAMeLBERT Vneural:", X_camel.shape)

CAMeLBERT Vneural: (7101, 768)


## 4 · XLM-R embeddings — load if cached, else build inline

In [4]:
xlmr = load_emb(P_XLMR_EMB, W_XLMR_EMB)
if xlmr is None:
    import torch
    from transformers import AutoTokenizer, AutoModel
    from tqdm.auto import tqdm
    tok = AutoTokenizer.from_pretrained(XLMR_MODEL)
    model = AutoModel.from_pretrained(XLMR_MODEL).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; model.to(dev)
    cls_id = tok.cls_token_id     # XLM-R <s>
    sep_id = tok.sep_token_id     # XLM-R </s>
    @torch.no_grad()
    def embed(text, max_ct=510, stride=460):
        ids = tok(str(text), add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
        chunks = [ids[i:i+max_ct] for i in range(0, len(ids), stride)] or [ids]
        vecs = []
        for c in chunks:
            inp = torch.tensor([[cls_id] + c + [sep_id]], device=dev)
            att = torch.ones_like(inp)
            out = model(input_ids=inp, attention_mask=att).last_hidden_state[:, 0, :]   # <s>
            vecs.append(out.squeeze(0).cpu().numpy())
        return np.mean(vecs, axis=0).astype(np.float32)
    M = np.vstack([embed(t) for t in tqdm(df["text"].tolist(), desc="XLM-R <s>")])
    xlmr = pd.DataFrame(M, index=df.index, columns=[f"x{i}" for i in range(768)])
    xlmr.reset_index().to_parquet(W_XLMR_EMB, index=False)
    print("built + saved ->", W_XLMR_EMB)
else:
    print("loaded cached XLM-R embeddings")
XLMR_COLS = [c for c in xlmr.columns if c != "article_id"]
assert len(XLMR_COLS)==768 and (xlmr.index==df.index).all()
X_xlmr = xlmr[XLMR_COLS].to_numpy(np.float32)
print("XLM-R Vneural:", X_xlmr.shape)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-R <s>:   0%|          | 0/7101 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1235 > 512). Running this sequence through the model will result in indexing errors


built + saved -> /kaggle/working/xlmr_embeddings.parquet
XLM-R Vneural: (7101, 768)


## 5 · Probe harness (frozen: raw embeddings + LogReg), NB5c protocol

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
LOGREG_KW = dict(max_iter=2000, class_weight="balanced")

def probe(X):
    # std: fit train, eval test (macro-F1). LOGO: fit (train+val) minus held-out gen AI,
    # eval test-split held-out gen AI + all test human. Raw embeddings (NB5c convention).
    clf = LogisticRegression(**LOGREG_KW).fit(X[is_train], y[is_train])
    std = 100*f1_score(y[is_test], clf.predict(X[is_test]), average="macro")
    human = (y==0); per = {}
    for g in GENERATORS:
        tr = is_tv   & (human | (gen != g))
        te = is_test & (human | (gen == g))
        c = LogisticRegression(**LOGREG_KW).fit(X[tr], y[tr])
        per[g] = 100*f1_score(y[te], c.predict(X[te]), average="macro")
    v = np.array(list(per.values()))
    return round(std,2), round(v.mean(),2), round(v.min(),2), {g:round(per[g],1) for g in GENERATORS}
print("harness ready")

harness ready


## 6 · Run XLM-R + CAMeLBERT (same harness) and compare

In [6]:
camel_std, camel_m, camel_w, camel_per = probe(X_camel)
xlmr_std,  xlmr_m,  xlmr_w,  xlmr_per  = probe(X_xlmr)

print("Per-generator LOGO (this harness):")
print(f"  CAMeLBERT-MSA  " + "  ".join(f"{g}:{camel_per[g]}" for g in GENERATORS))
print(f"  XLM-R          " + "  ".join(f"{g}:{xlmr_per[g]}"  for g in GENERATORS))

rows = []
for name,(s,m,w) in NB5C.items():
    rows.append({"encoder":name, "source":"NB5c(ref)", "std":s, "LOGO_mean":m, "LOGO_worst":w})
rows.append({"encoder":"CAMeLBERT-MSA", "source":"this-harness", "std":camel_std, "LOGO_mean":camel_m, "LOGO_worst":camel_w})
rows.append({"encoder":"XLM-R",         "source":"this-harness", "std":xlmr_std,  "LOGO_mean":xlmr_m,  "LOGO_worst":xlmr_w})
res = pd.DataFrame(rows)
import IPython.display as ipd; ipd.display(res)

Per-generator LOGO (this harness):
  CAMeLBERT-MSA  deepseek:99.1  sonnet:99.5  qwen:98.3  gemini:100.0  gpt:91.7  opus:100.0
  XLM-R          deepseek:92.1  sonnet:95.4  qwen:90.3  gemini:91.4  gpt:90.3  opus:90.9


,encoder,source,std,LOGO_mean,LOGO_worst
0,AraBERTv2,NB5c(ref),99.50,97.60,92.80
1,AraELECTRA,NB5c(ref),99.50,96.20,83.50
2,CAMeLBERT-MSA,NB5c(ref),99.70,98.10,92.70
3,CAMeLBERT-MSA,this-harness,99.73,98.10,91.68
4,XLM-R,this-harness,96.98,91.75,90.33


## 7 · Verdict

In [7]:
print("="*62)
print("FAIR comparison (same harness):  XLM-R vs CAMeLBERT-MSA")
print(f"  std        XLM-R {xlmr_std:5}  |  CAMeLBERT {camel_std:5}")
print(f"  LOGO-mean  XLM-R {xlmr_m:5}  |  CAMeLBERT {camel_m:5}")
print(f"  LOGO-worst XLM-R {xlmr_w:5}  |  CAMeLBERT {camel_w:5}   (delta {xlmr_w-camel_w:+.2f}pp, 1 gpt art ~0.42)")
print("="*62)
if xlmr_w > camel_w + 0.42 and xlmr_m > camel_m:
    print("=> XLM-R beats CAMeLBERT-MSA on the robustness floor -> reconsider the neural core (report both).")
elif camel_w > xlmr_w + 0.42:
    print("=> CAMeLBERT-MSA still wins the floor -> core choice JUSTIFIED with evidence; XLM-R added as probe.")
else:
    print("=> Within noise on the floor -> CAMeLBERT-MSA stays (domain-pretrained, prior choice); note XLM-R parity.")
print("Sanity: this-harness CAMeLBERT-MSA should sit within ~0.5pp of its NB5c row (99.7/98.1/92.7).")

FAIR comparison (same harness):  XLM-R vs CAMeLBERT-MSA
  std        XLM-R 96.98  |  CAMeLBERT 99.73
  LOGO-mean  XLM-R 91.75  |  CAMeLBERT  98.1
  LOGO-worst XLM-R 90.33  |  CAMeLBERT 91.68   (delta -1.35pp, 1 gpt art ~0.42)
=> CAMeLBERT-MSA still wins the floor -> core choice JUSTIFIED with evidence; XLM-R added as probe.
Sanity: this-harness CAMeLBERT-MSA should sit within ~0.5pp of its NB5c row (99.7/98.1/92.7).
